# 507 虚频修正（vaspkit 功能 507）

## 功能说明

1. **输入**：HAS_IMAG 结构目录列表（来自 tool_03 结果解析）
2. **执行**：在每个目录下运行 vaspkit 507，校正因子 **0.01**
3. **输出**：POSCAR_NEW 覆盖 POSCAR，可重提频率任务

## 分支判定
- 全部虚频 < 50 cm⁻¹：自动校正
- 存在虚频 ≥ 50 cm⁻¹：本 main 为 final-purpose，选 `no` 校正所有虚频

## 使用方式
- 修改下方 `has_imag_dirs` 或 `zpe_root` 扫描模式
- 依次运行各 Cell

In [ ]:
import os
import re
import subprocess
import shutil
from pathlib import Path

In [ ]:
# ============================================
# 从 shared/config.py 读取 zpe_root（clio 可直接修改 config.py）
# ============================================
import sys
from pathlib import Path
_p = Path.cwd()
while _p != _p.parent:
    if (_p / "shared" / "config.py").exists():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
from shared.load_config import load_config
cfg = load_config()
zpe_root = cfg.ZPE_ROOT

# 校正因子（运行规则规定 0.01）
CORRECTION_FACTOR = "0.01"

# 方式1：直接指定 HAS_IMAG 目录列表（覆盖 zpe_root 扫描）
has_imag_dirs = []

In [ ]:
def read_imaginary_freqs(outcar_path):
    """从 OUTCAR 读取虚频列表（cm^-1），用于判断是否需 507 及记录。"""
    if not os.path.exists(outcar_path):
        return []
    freq_pattern = r'(-?\d+\.?\d*(?:[eE][+-]?\d+)?)\s+cm-1'
    imaginary_freqs = []
    with open(outcar_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if "THz" in line and ("cm^-1" in line or "cm-1" in line):
                matches = re.findall(freq_pattern, line)
                for m in matches:
                    try:
                        freq = float(m)
                        if freq < 0 or "f/i" in line:
                            imaginary_freqs.append(abs(freq))
                    except ValueError:
                        pass
    return list(dict.fromkeys(imaginary_freqs))  # 去重保序


def run_vaspkit_507(work_dir, factor="0.01"):
    """
    在 work_dir 下运行 vaspkit 507。
    vaspkit 507 交互：若 max_imag>=50 会问 "Type yes or no!" -> no；然后问因子 -> factor
    全部<50 时可能只问因子。为兼容两种情形，先送 no\n 再送 factor。
    """
    inp = f"no\n{factor}\n"
    proc = subprocess.Popen(
        ["vaspkit"],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        cwd=work_dir,
        text=True,
    )
    # vaspkit 启动后：若从主菜单选 507，则首行输入 507；再根据提示输入 no 和因子
    # 交互顺序：max>=50 时先 "Type yes or no!" -> no；然后 "input factor" -> factor
    # 若本机 vaspkit 菜单不同，请按实际调整 full_inp
    full_inp = f"507\n{inp}"
    out, err = proc.communicate(input=full_inp, timeout=60)
    return proc.returncode == 0, out, err


def scan_has_imag_from_zpe(zpe_root):
    """扫描 ZPE 目录，找出含 OUTCAR 且存在虚频的目录。"""
    dirs_with_imag = []
    zpe = Path(zpe_root)
    if not zpe.exists():
        return dirs_with_imag
    for body in ["1_body", "2_body"]:
        body_path = zpe / body
        if not body_path.exists():
            continue
        for d in body_path.rglob("*"):
            if d.is_dir():
                outcar = d / "OUTCAR"
                if outcar.exists():
                    imags = read_imaginary_freqs(str(outcar))
                    if imags:
                        dirs_with_imag.append(str(d))
    return dirs_with_imag

In [ ]:
# 确定待修正目录列表
if zpe_root and Path(zpe_root).exists():
    to_fix = scan_has_imag_from_zpe(zpe_root)
    print(f"从 {zpe_root} 扫描到 {len(to_fix)} 个 HAS_IMAG 目录")
else:
    to_fix = [d for d in has_imag_dirs if os.path.isdir(d)]
    print(f"使用手动指定的 {len(to_fix)} 个目录")

if not to_fix:
    print("无待修正目录，请检查 has_imag_dirs 或 zpe_root。")
else:
    for d in to_fix:
        imags = read_imaginary_freqs(os.path.join(d, "OUTCAR"))
        print(f"  - {d}: 虚频 {len(imags)} 个, {[f'{x:.1f}' for x in imags[:5]]} cm^-1")

In [ ]:
# 执行 507 修正
results = []
for work_dir in to_fix:
    outcar = os.path.join(work_dir, "OUTCAR")
    poscar_new = os.path.join(work_dir, "POSCAR_NEW")
    poscar = os.path.join(work_dir, "POSCAR")
    
    if not os.path.exists(outcar):
        print(f"⚠ {work_dir}: 无 OUTCAR，跳过")
        results.append((work_dir, "SKIP", "无OUTCAR"))
        continue
    
    imags = read_imaginary_freqs(outcar)
    print(f"\n>>> 处理: {work_dir}")
    print(f"    原虚频: {[f'{x:.2f}' for x in imags]} cm^-1")
    
    ok, out, err = run_vaspkit_507(work_dir, CORRECTION_FACTOR)
    
    if not ok:
        print(f"    ✗ vaspkit 507 执行异常: {err[:200] if err else out[-200:]}")
        results.append((work_dir, "FAIL", err or out))
        continue
    
    if not os.path.exists(poscar_new):
        print(f"    ✗ 未生成 POSCAR_NEW")
        results.append((work_dir, "FAIL", "无POSCAR_NEW"))
        continue
    
    shutil.copy(poscar_new, poscar)
    Path(os.path.join(work_dir, "_507_corrected")).touch()  # 标记已修正，供最终结构提取识别
    print(f"    ✓ POSCAR_NEW 已覆盖 POSCAR")
    results.append((work_dir, "OK", f"因子{CORRECTION_FACTOR}"))

print("\n--- 汇总 ---")
for d, status, msg in results:
    print(f"  {status}: {os.path.basename(d)} - {msg}")

## 后续步骤

修正完成后，使用 **tool_02** 的提交逻辑重新提交频率任务：
- 进入对应 ZPE 根目录
- 运行 `tool_02_提交调度/auto/frequency.ipynb` 或 `qsub submit.sh`